# EVT (padronizado)


*By Miguel Ferreira*

**Este notebook, assim com todos os outros de cada ferramenta do envelope de risco, segue o mesmo protocolo:**
1. Importação do dataset e bibliotecas
2. Execução do ```setup()``` e alinhamento temporal
3. Construção da ferramenta de risco 
4. Sanity checks mínimos
5. DataFrame final (features de risco) 
6. Padronização
7. Salvamento

Esta padronização é uma peça fundamental para um projeto **clean code.** Tanto que esta introdução estará presente em todos os notebooks de todas as ferramentas do envelope de risco.

---


## 1. Importação do dataset e bibliotecas


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import genpareto

sys.path.append(str(Path("../../src").resolve()))
from setup import setup

CSV_PATH = "../../data/processed/financial_tools_datset.csv"


## 2. Execução do `setup()` e alinhamento temporal


In [2]:
raw_df = pd.read_csv(CSV_PATH)

data = setup(CSV_PATH)

X_train = data["X_train"]
X_val = data["X_val"]
X_test = data["X_test"]

df = raw_df.copy()
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df = df.sort_values("Date").set_index("Date")
returns = df["Price"].pct_change()


## 3. Construção da ferramenta de risco


In [3]:
window = 100
threshold_q = 0.95
alpha = 0.99


def fit_evt(series, threshold_q):
    threshold = np.quantile(series, threshold_q)
    excesses = series[series > threshold] - threshold

    if len(excesses) < 5:
        return np.nan, np.nan

    c, loc, scale = genpareto.fit(excesses)
    return c, scale


evt_shape = []
evt_scale = []
evt_var = []
evt_cvar = []

for i in range(len(returns)):
    if i < window:
        evt_shape.append(np.nan)
        evt_scale.append(np.nan)
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue

    window_data = returns.iloc[i - window : i].dropna()

    if len(window_data) < window:
        evt_shape.append(np.nan)
        evt_scale.append(np.nan)
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue

    xi, beta = fit_evt(window_data, threshold_q)

    evt_shape.append(xi)
    evt_scale.append(beta)

    if np.isnan(xi) or np.isnan(beta):
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue

    threshold = np.quantile(window_data, threshold_q)

    var_evt = threshold + (beta / xi) * ((1 - alpha) ** (-xi) - 1)
    cvar_evt = (var_evt + (beta - xi * threshold)) / (1 - xi)

    evt_var.append(var_evt)
    evt_cvar.append(cvar_evt)

evt_df_check = pd.DataFrame(
    {
        "evt_shape": evt_shape,
        "evt_scale": evt_scale,
        "evt_var": evt_var,
        "evt_cvar": evt_cvar,
    },
    index=df.index,
)


## 4. Sanity checks mínimos


In [4]:
print("NaN críticos:")
print(evt_df_check.isna().sum())

print("\nAlinhamento temporal:")
print("Index monotonic increasing:", evt_df_check.index.is_monotonic_increasing)
print("Index has duplicates:", evt_df_check.index.has_duplicates)

print("\nDistribuição plausível:")
print(evt_df_check.describe().T[["mean", "std", "min", "max"]])


NaN críticos:
evt_shape    101
evt_scale    101
evt_var      101
evt_cvar     101
dtype: int64

Alinhamento temporal:
Index monotonic increasing: True
Index has duplicates: False

Distribuição plausível:
               mean       std       min        max
evt_shape  1.218147  0.332048  0.439059   1.639289
evt_scale  0.000854  0.000661  0.000076   0.003644
evt_var    0.249168  0.204045  0.010891   1.592355
evt_cvar   0.381355  4.426486 -4.097976  21.885639


## 5. DataFrame final (features de risco)


In [5]:
evt_dataset = pd.DataFrame(index=df.index)

evt_dataset["evt_shape"] = evt_shape
evt_dataset["evt_scale"] = evt_scale
evt_dataset["evt_var"] = evt_var
evt_dataset["evt_cvar"] = evt_cvar


## 6. Padronização


In [6]:
evt_dataset = evt_dataset.sort_index().bfill()
evt_dataset.index.name = "timestamp"
evt_dataset = evt_dataset.astype(np.float32)

evt_dataset.tail()


,evt_shape,evt_scale,evt_var,evt_cvar
timestamp,,,,
2026-02-06,1.612584,0.000197,0.210060,-0.330263
2026-02-08,1.612584,0.000197,0.210060,-0.330263
2026-02-09,1.612584,0.000197,0.210060,-0.330263
2026-02-10,1.421219,0.000333,0.168933,-0.381802
2026-02-11,1.421219,0.000333,0.168933,-0.381802


## 7. Salvamento


In [7]:
'''
OUTPUT_PATH = "../../data/processed/evt_features.parquet"
evt_dataset.to_parquet(OUTPUT_PATH, index=True)
OUTPUT_PATH
'''


'\nOUTPUT_PATH = "../../data/processed/evt_features.parquet"\nevt_dataset.to_parquet(OUTPUT_PATH, index=True)\nOUTPUT_PATH\n'